# Boruta at Scale: Shadow Features, Statistical Testing, and Optuna

## Learning Objectives

By completing this notebook, you will:
1. Run Boruta with LightGBM as the base estimator on a real classification dataset
2. Visualise shadow feature importance distributions vs real feature distributions
3. Compare Boruta's all-relevant set with the SFS minimal-optimal set from Notebook 01
4. Implement Optuna TPE as an alternative wrapper selector and compare to Boruta
5. Measure how both methods scale as feature count increases

## Prerequisites
- Notebook 01: Sequential selection (SFS/SFFS results for comparison)
- Guide 02: Boruta and advanced wrappers theory
- LightGBM installation: `pip install lightgbm optuna`

## Estimated Time: 15 minutes

---

## 1. Setup and Dataset

We use the **breast cancer Wisconsin dataset** (30 features) for all core analyses. For the scaling experiment we generate synthetic datasets with increasing feature counts (30, 60, 120) to measure wall time growth.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from scipy.stats import binom

try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
    print('LightGBM available.')
except ImportError:
    LGBM_AVAILABLE = False
    from sklearn.ensemble import RandomForestClassifier
    print('LightGBM not installed — using RandomForest as fallback.')

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
    print('Optuna available.')
except ImportError:
    OPTUNA_AVAILABLE = False
    print('Optuna not installed — Optuna section will be skipped.')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Load breast cancer dataset: 569 samples, 30 features
bc = load_breast_cancer()
X = bc.data.astype(float)
y = bc.target
feature_names = list(bc.feature_names)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Class balance: {y.mean():.1%} malignant')
print(f'Features: {feature_names[:5]} ... ({len(feature_names)} total)')

In [ ]:
# Create base LightGBM estimator for Boruta
# Settings mirror those recommended for Boruta: subsample for bootstrap behaviour
if LGBM_AVAILABLE:
    BORUTA_ESTIMATOR = lgb.LGBMClassifier(
        n_estimators=200,
        max_depth=5,
        subsample=0.632,           # Bootstrap fraction matching RF default
        colsample_bytree=0.632,
        learning_rate=0.05,
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
    )
else:
    # Fallback to Random Forest if LightGBM not available
    from sklearn.ensemble import RandomForestClassifier
    BORUTA_ESTIMATOR = RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        max_samples=0.632,
        random_state=42,
        n_jobs=-1,
    )

print(f'Base estimator: {type(BORUTA_ESTIMATOR).__name__}')

## 2. Boruta Implementation

We implement Boruta from scratch to make the shadow feature mechanism fully visible. The key loop: create shadow copies → augment matrix → train model → compare real vs shadow importances → run binomial test → update feature status.

In [ ]:
def run_boruta(
    X, y, estimator,
    max_iter=100,
    alpha=0.05,
    percentile=100,
    verbose=True,
    seed=42,
):
    """
    Boruta feature selection.

    For each iteration:
      1. Permute each column of X to create shadow features (zero information)
      2. Augment: [X | X_shadow]
      3. Fit base estimator, extract feature importances
      4. Threshold = max importance among shadow features (MZSA)
      5. Real feature hits +1 if importance > threshold
      6. Binomial test (Bonferroni corrected) to decide confirmed/rejected

    Parameters
    ----------
    X : array (n, p)
    y : array (n,)
    estimator : tree-based sklearn estimator
    max_iter : int
    alpha : float   Family-wise error rate
    percentile : int  Percentile of shadow importances as threshold (100=max)
    verbose : bool
    seed : int

    Returns
    -------
    confirmed : ndarray of int   Confirmed relevant feature indices
    rejected : ndarray of int    Confirmed irrelevant feature indices
    tentative : ndarray of int   Undecided feature indices
    importance_history : ndarray (max_iter, p)  Per-iteration importances
    shadow_max_history : list     MZSA at each iteration
    """
    rng = np.random.default_rng(seed)
    n, p = X.shape

    hits = np.zeros(p, dtype=int)
    status = np.zeros(p, dtype=int)  # 0=tentative, 1=confirmed, -1=rejected
    importance_history = np.full((max_iter, p), np.nan)
    shadow_max_history = []
    actual_iters = 0

    for t in range(max_iter):
        # Tentative features still being tested
        tentative_mask = (status == 0)
        if not tentative_mask.any():
            break
        actual_iters = t + 1

        # Step 1: Create shadow matrix (permute each column independently)
        X_shadow = np.apply_along_axis(rng.permutation, 0, X)

        # Step 2: Augment feature matrix
        X_aug = np.hstack([X, X_shadow])  # shape (n, 2p)

        # Step 3: Fit estimator and extract importances
        model = clone(estimator)
        model.fit(X_aug, y)
        importances = np.array(model.feature_importances_)

        real_imp = importances[:p]
        shadow_imp = importances[p:]

        # Step 4: Shadow threshold (default: max = 100th percentile)
        shadow_thresh = np.percentile(shadow_imp, percentile)
        shadow_max_history.append(shadow_thresh)
        importance_history[t, :] = real_imp

        # Step 5: Update hits for tentative features
        tentative_indices = np.where(tentative_mask)[0]
        for j in tentative_indices:
            if real_imp[j] > shadow_thresh:
                hits[j] += 1

        # Step 6: Binomial test with Bonferroni correction
        n_tentative = len(tentative_indices)
        if n_tentative > 0:
            bonferroni_threshold = alpha / (2 * n_tentative)

            for j in tentative_indices:
                h = hits[j]
                # Upper tail: P(X >= h) under Bin(t+1, 0.5)
                p_upper = 1 - binom.cdf(h - 1, t + 1, 0.5)
                # Lower tail: P(X <= h) under Bin(t+1, 0.5)
                p_lower = binom.cdf(h, t + 1, 0.5)

                if p_upper < bonferroni_threshold:
                    status[j] = 1   # Confirmed relevant
                elif p_lower < bonferroni_threshold:
                    status[j] = -1  # Confirmed irrelevant

        if verbose and (t + 1) % 20 == 0:
            n_conf = (status == 1).sum()
            n_rej = (status == -1).sum()
            n_tent = (status == 0).sum()
            print(f'  Iter {t+1:3d}: confirmed={n_conf}, rejected={n_rej}, tentative={n_tent}')

    confirmed = np.where(status == 1)[0]
    rejected = np.where(status == -1)[0]
    tentative = np.where(status == 0)[0]

    return confirmed, rejected, tentative, importance_history[:actual_iters], shadow_max_history


print('Boruta function defined.')

In [ ]:
print('Running Boruta (100 iterations)...')
t0 = time.time()
confirmed, rejected, tentative, imp_hist, shadow_hist = run_boruta(
    X_scaled, y, BORUTA_ESTIMATOR, max_iter=100, alpha=0.05, verbose=True
)
boruta_time = time.time() - t0

print(f'\nBoruta completed in {boruta_time:.1f}s')
print(f'Confirmed relevant ({len(confirmed)}): {[feature_names[i] for i in confirmed]}')
print(f'Confirmed irrelevant ({len(rejected)}): {[feature_names[i] for i in rejected]}')
print(f'Tentative ({len(tentative)}): {[feature_names[i] for i in tentative]}')

## 3. Visualise Shadow vs Real Importance Distributions

For each real feature, plot the distribution of its importance scores across all iterations. Overlay the shadow feature threshold (dashed red line). Features whose distributions sit clearly above the threshold are confirmed relevant; features below are confirmed irrelevant.

In [ ]:
def plot_boruta_importances(imp_hist, shadow_hist, feature_names, confirmed, rejected, tentative):
    """
    Violin plot of per-feature importance distributions.
    Overlay median shadow max as dashed red line.
    Colour by Boruta decision: green=confirmed, red=rejected, grey=tentative.
    """
    p = len(feature_names)
    # Collect non-NaN importances per feature
    feat_data = [
        imp_hist[:, j][~np.isnan(imp_hist[:, j])]
        for j in range(p)
    ]

    # Sort by median importance (descending)
    order = np.argsort([np.median(d) for d in feat_data])[::-1]

    confirmed_set = set(confirmed)
    rejected_set = set(rejected)

    fig, ax = plt.subplots(figsize=(18, 5))

    parts = ax.violinplot(
        [feat_data[i] for i in order],
        positions=range(p),
        showmedians=True,
        showextrema=False,
    )

    # Colour by decision status
    for i, patch in enumerate(parts['bodies']):
        j = order[i]
        if j in confirmed_set:
            patch.set_facecolor('#27ae60')
            patch.set_alpha(0.75)
        elif j in rejected_set:
            patch.set_facecolor('#e74c3c')
            patch.set_alpha(0.75)
        else:  # tentative
            patch.set_facecolor('#95a5a6')
            patch.set_alpha(0.75)

    # Shadow max line
    median_shadow = np.median(shadow_hist)
    ax.axhline(
        median_shadow, color='#c0392b', linestyle='--', linewidth=2,
        label=f'Median shadow max = {median_shadow:.4f}'
    )

    ax.set_xticks(range(p))
    ax.set_xticklabels(
        [feature_names[i] for i in order],
        rotation=45, ha='right', fontsize=7
    )
    ax.set_ylabel('Feature importance (LightGBM split gain)', fontsize=11)
    ax.set_title('Boruta: Real Feature Importances vs Shadow Threshold', fontsize=12)

    legend_elements = [
        mpatches.Patch(facecolor='#27ae60', label=f'Confirmed relevant ({len(confirmed)})', alpha=0.75),
        mpatches.Patch(facecolor='#e74c3c', label=f'Confirmed irrelevant ({len(rejected)})', alpha=0.75),
        mpatches.Patch(facecolor='#95a5a6', label=f'Tentative ({len(tentative)})', alpha=0.75),
        plt.Line2D([0], [0], color='#c0392b', linestyle='--', linewidth=2,
                   label=f'Median shadow max = {median_shadow:.4f}'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_boruta_importances(
    imp_hist, shadow_hist, feature_names, confirmed, rejected, tentative
)

## 4. Boruta All-Relevant vs SFS Minimal-Optimal

Boruta returns the *all-relevant* set: every feature that carries any information. SFS returns a *minimal-optimal* set: the smallest subset achieving near-best performance. Here we compare them directly.

In [ ]:
# Run SFS to k* = len(confirmed) features for a fair comparison
from sklearn.model_selection import cross_val_score

K_TARGET_COMPARISON = max(len(confirmed), 5)  # At least 5 features

print(f'Running SFS to k* = {K_TARGET_COMPARISON} (same as Boruta confirmed count)...')

# Fast SFS using sklearn's implementation for speed
from sklearn.feature_selection import SequentialFeatureSelector

if LGBM_AVAILABLE:
    sfs_estimator = lgb.LGBMClassifier(
        n_estimators=50, max_depth=4, verbosity=-1, random_state=42
    )
else:
    from sklearn.ensemble import RandomForestClassifier
    sfs_estimator = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42)

sfs_selector = SequentialFeatureSelector(
    sfs_estimator,
    n_features_to_select=K_TARGET_COMPARISON,
    direction='forward',
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
)

t0 = time.time()
sfs_selector.fit(X_scaled, y)
sfs_time = time.time() - t0

sfs_selected = np.where(sfs_selector.support_)[0]

print(f'SFS completed in {sfs_time:.1f}s')
print(f'SFS selected ({len(sfs_selected)}): {[feature_names[i] for i in sfs_selected]}')

In [ ]:
# Compare the two sets
boruta_set = set(confirmed)
sfs_set = set(sfs_selected.tolist())

in_both = boruta_set & sfs_set
only_boruta = boruta_set - sfs_set
only_sfs = sfs_set - boruta_set

print(f'\nFeature set comparison (Boruta confirmed vs SFS k*={K_TARGET_COMPARISON}):')
print(f'  In both:      {sorted(in_both)} = {[feature_names[i] for i in sorted(in_both)]}')
print(f'  Boruta only:  {sorted(only_boruta)} = {[feature_names[i] for i in sorted(only_boruta)]}')
print(f'  SFS only:     {sorted(only_sfs)} = {[feature_names[i] for i in sorted(only_sfs)]}')
print(f'  Overlap: {len(in_both)}/{max(len(boruta_set), len(sfs_set))} = {len(in_both)/max(len(boruta_set),len(sfs_set)):.0%}')

In [ ]:
# Head-to-head test accuracy comparison
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

if LGBM_AVAILABLE:
    eval_model = lgb.LGBMClassifier(n_estimators=200, verbosity=-1, random_state=42)
else:
    from sklearn.ensemble import RandomForestClassifier
    eval_model = RandomForestClassifier(n_estimators=200, random_state=42)

results_comparison = {}

# All features baseline
eval_model.fit(X_train, y_train)
results_comparison['All features (30)'] = eval_model.score(X_test, y_test)

# Boruta confirmed
if len(confirmed) > 0:
    eval_model.fit(X_train[:, confirmed], y_train)
    results_comparison[f'Boruta confirmed ({len(confirmed)})'] = eval_model.score(X_test[:, confirmed], y_test)

# SFS
eval_model.fit(X_train[:, sfs_selected], y_train)
results_comparison[f'SFS k*={K_TARGET_COMPARISON}'] = eval_model.score(X_test[:, sfs_selected], y_test)

print('Test accuracy comparison:')
for method, score in results_comparison.items():
    print(f'  {method:<35} {score:.4f}')

In [ ]:
# Venn-style bar chart showing feature membership
p = len(feature_names)

fig, ax = plt.subplots(figsize=(14, 4))

bar_positions = np.arange(p)
bar_width = 0.35

# Boruta confirmed: 1, else 0
boruta_indicator = np.array([1 if j in boruta_set else 0 for j in range(p)])
sfs_indicator = np.array([1 if j in sfs_set else 0 for j in range(p)])

ax.bar(bar_positions - bar_width/2, boruta_indicator, bar_width,
       label=f'Boruta confirmed ({len(confirmed)})',
       color='#27ae60', alpha=0.8)
ax.bar(bar_positions + bar_width/2, sfs_indicator, bar_width,
       label=f'SFS k*={K_TARGET_COMPARISON}',
       color='#3498db', alpha=0.8)

ax.set_xticks(bar_positions)
ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=7)
ax.set_yticks([0, 1])
ax.set_yticklabels(['Not selected', 'Selected'])
ax.set_title('Boruta (All-Relevant) vs SFS (Minimal-Optimal): Feature Membership', fontsize=12)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Optuna TPE as Alternative Wrapper

Instead of greedy sequential search, Optuna TPE builds a probabilistic model of which feature combinations yield high scores. Each trial samples a binary feature mask, evaluates it with cross-validation, and updates the model.

In [ ]:
if OPTUNA_AVAILABLE:
    from sklearn.model_selection import cross_val_score
    from sklearn.base import clone

    p = X_scaled.shape[1]
    MIN_FEATURES = 3
    N_TRIALS = 150  # Reduced for notebook speed; use 300+ in production

    if LGBM_AVAILABLE:
        optuna_base = lgb.LGBMClassifier(
            n_estimators=100, max_depth=4, verbosity=-1, random_state=42
        )
    else:
        from sklearn.ensemble import RandomForestClassifier
        optuna_base = RandomForestClassifier(
            n_estimators=100, max_depth=4, random_state=42
        )

    optuna_cache = {}

    def objective(trial):
        # Sample a binary mask: suggest_categorical returns True/False per feature
        mask = np.array(
            [trial.suggest_categorical(f'f{j}', [True, False]) for j in range(p)]
        )

        if mask.sum() < MIN_FEATURES:
            return -np.inf  # Penalise trivially small subsets

        key = tuple(np.where(mask)[0])
        if key in optuna_cache:
            return optuna_cache[key]

        scores = cross_val_score(
            clone(optuna_base), X_scaled[:, key], y, cv=5, scoring='accuracy'
        )
        result = float(scores.mean())
        optuna_cache[key] = result
        return result

    print(f'Running Optuna TPE ({N_TRIALS} trials)...')
    t0 = time.time()
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
    optuna_time = time.time() - t0

    # Extract best feature mask
    best_params = study.best_params
    optuna_mask = np.array([best_params[f'f{j}'] for j in range(p)])
    optuna_selected = np.where(optuna_mask)[0]

    print(f'Optuna completed in {optuna_time:.1f}s  |  Cache hits: {len(optuna_cache)} unique evaluations')
    print(f'Best trial score: {study.best_value:.4f}')
    print(f'Selected ({len(optuna_selected)} features): {[feature_names[i] for i in optuna_selected]}')

else:
    print('Optuna not available — install with: pip install optuna')
    optuna_selected = confirmed  # Use Boruta as fallback
    study = None

In [ ]:
if OPTUNA_AVAILABLE and study is not None:
    # Plot Optuna optimisation history
    trial_values = [t.value for t in study.trials if t.value is not None and t.value > -np.inf]
    running_best = np.maximum.accumulate(trial_values)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Panel 1: Optimisation history
    axes[0].scatter(range(len(trial_values)), trial_values,
                    alpha=0.4, s=15, color='#3498db', label='Trial score')
    axes[0].plot(range(len(running_best)), running_best,
                 color='#e74c3c', linewidth=2, label='Running best')
    axes[0].set_xlabel('Trial number')
    axes[0].set_ylabel('CV Accuracy')
    axes[0].set_title('Optuna TPE: Optimisation History')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Panel 2: Feature selection frequency across top-20% trials
    top_threshold = np.percentile(trial_values, 80)
    top_trials = [
        t for t in study.trials
        if t.value is not None and t.value >= top_threshold
    ]

    feature_freq = np.zeros(p)
    for t in top_trials:
        mask = np.array([t.params.get(f'f{j}', False) for j in range(p)])
        feature_freq += mask

    feature_freq /= max(len(top_trials), 1)
    order_freq = np.argsort(feature_freq)[::-1][:20]  # Top 20 by frequency

    axes[1].barh(
        [feature_names[i] for i in order_freq[::-1]],
        feature_freq[order_freq[::-1]],
        color='#9b59b6', alpha=0.8
    )
    axes[1].axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='50% frequency')
    axes[1].set_xlabel('Selection frequency in top-20% trials')
    axes[1].set_title('Feature Selection Frequency (Optuna Top Trials)')
    axes[1].legend()
    axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f'Top-20% trials: {len(top_trials)} trials with score >= {top_threshold:.4f}')

## 6. Scaling Behaviour with Increasing Feature Count

Measure how Boruta and SFS wall time grows as the number of features increases. This reveals when each method becomes impractical and when to switch strategies.

In [ ]:
from sklearn.datasets import make_classification

feature_counts = [20, 50, 100]
boruta_times = []
sfs_times = []

for n_features in feature_counts:
    print(f'\np = {n_features} features:')

    # Generate synthetic dataset: 500 samples, n_features features
    # n_informative = 10 (fixed), rest are noise
    X_syn, y_syn = make_classification(
        n_samples=500,
        n_features=n_features,
        n_informative=min(10, n_features // 2),
        n_redundant=min(5, n_features // 4),
        random_state=42,
        n_clusters_per_class=2,
    )
    X_syn = StandardScaler().fit_transform(X_syn)

    # --- Boruta timing: 30 iterations (reduced for speed) ---
    if LGBM_AVAILABLE:
        boruta_est = lgb.LGBMClassifier(
            n_estimators=100, max_depth=4, verbosity=-1, random_state=42
        )
    else:
        boruta_est = RandomForestClassifier(
            n_estimators=100, max_depth=4, random_state=42, n_jobs=-1
        )

    t0 = time.time()
    run_boruta(X_syn, y_syn, boruta_est, max_iter=30, verbose=False)
    bt = time.time() - t0
    boruta_times.append(bt)
    print(f'  Boruta (30 iters):  {bt:.2f}s')

    # --- SFS timing: k*=10 ---
    if LGBM_AVAILABLE:
        sfs_est = lgb.LGBMClassifier(
            n_estimators=50, max_depth=3, verbosity=-1, random_state=42
        )
    else:
        sfs_est = RandomForestClassifier(
            n_estimators=50, max_depth=3, random_state=42
        )

    sfs_sel = SequentialFeatureSelector(
        sfs_est, n_features_to_select=min(10, n_features // 2),
        direction='forward', scoring='accuracy', cv=3, n_jobs=-1
    )

    t0 = time.time()
    sfs_sel.fit(X_syn, y_syn)
    st = time.time() - t0
    sfs_times.append(st)
    print(f'  SFS (k*=10, cv=3):  {st:.2f}s')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw times
axes[0].plot(feature_counts, boruta_times, 'o-', color='#27ae60', linewidth=2,
             markersize=8, label='Boruta (30 iters)')
axes[0].plot(feature_counts, sfs_times, 's-', color='#3498db', linewidth=2,
             markersize=8, label='SFS (k*=10, cv=3)')
axes[0].set_xlabel('Number of features (p)')
axes[0].set_ylabel('Wall time (seconds)')
axes[0].set_title('Scaling: Boruta vs SFS')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Time per feature (normalised)
boruta_per_feat = [t / p for t, p in zip(boruta_times, feature_counts)]
sfs_per_feat = [t / p for t, p in zip(sfs_times, feature_counts)]

axes[1].plot(feature_counts, boruta_per_feat, 'o-', color='#27ae60', linewidth=2,
             markersize=8, label='Boruta (per feature)')
axes[1].plot(feature_counts, sfs_per_feat, 's-', color='#3498db', linewidth=2,
             markersize=8, label='SFS (per feature)')
axes[1].set_xlabel('Number of features (p)')
axes[1].set_ylabel('Time per feature (seconds)')
axes[1].set_title('Cost per Feature: Boruta vs SFS')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Wall Time Scaling with Feature Count', fontsize=13)
plt.tight_layout()
plt.show()

print('\nScaling analysis:')
print('Boruta cost grows as O(T × p × T_RF) — linear in p for fixed T.')
print('SFS cost grows as O(k* × p × v × T_M) — also linear in p.')
print('SFS becomes faster than Boruta when k* × v < T (fixed iterations).')

## 7. Exercise: Compare Boruta's Tentative Features

**Task:** Boruta leaves some features as `tentative` — the statistical test did not reach significance in either direction. Run Boruta again with `max_iter=200` and observe whether the tentative features resolve. Plot the hit count history for all tentative features from the 100-iteration run to understand why they remained undecided.

**Expected output:** A line plot showing the cumulative hit count for each tentative feature over 100 iterations, overlaid with the expected hit count under H0 (T/2) and the Bonferroni upper/lower bounds.

In [ ]:
# YOUR CODE HERE
# ---------------
# Step 1: Modify run_boruta to return per-feature hit counts at each iteration
# Hint: store hits_per_iter as a 2D array (max_iter, p) where
#       hits_per_iter[t, j] = cumulative hits for feature j after iteration t

def run_boruta_with_hit_history(X, y, estimator, max_iter=100, alpha=0.05, seed=42):
    """
    Run Boruta and return the cumulative hit count history per feature.

    Returns
    -------
    confirmed, rejected, tentative : ndarray
    hit_history : ndarray (max_iter, p)  cumulative hits at each iteration
    """
    # TODO: implement
    pass

In [ ]:
# AUTO-GRADED TEST — do not modify
def test_hit_history():
    assert run_boruta_with_hit_history is not None, 'Define run_boruta_with_hit_history'
    result = run_boruta_with_hit_history(
        X_scaled[:100], y[:100], BORUTA_ESTIMATOR, max_iter=10, seed=0
    )
    assert result is not None, 'Function must return a result'
    assert len(result) == 4, 'Must return (confirmed, rejected, tentative, hit_history)'
    confirmed_t, rejected_t, tentative_t, hit_hist = result
    assert hit_hist is not None, 'hit_history must not be None'
    assert hit_hist.shape[1] == X_scaled.shape[1], (
        f'hit_history must have {X_scaled.shape[1]} columns, got {hit_hist.shape[1]}'
    )
    print(f'Test passed! hit_history shape: {hit_hist.shape}')

test_hit_history()

## Summary

### Key Takeaways

1. **Boruta and SFS answer different questions:** Boruta asks "is there signal?" (statistical test); SFS asks "does adding this feature help the model right now?" (greedy CV).
2. **Shadow features provide a principled noise baseline:** permuting each column destroys information while preserving the marginal distribution. A real feature that cannot beat the best shadow is indistinguishable from noise.
3. **Boruta's all-relevant set includes redundant features:** two correlated features that both carry signal will both be confirmed. SFS selects only one.
4. **Optuna TPE treats feature selection as hyperparameter optimisation:** it learns a probabilistic model of which feature combinations work, enabling it to find non-obvious combinations that sequential greedy search misses.
5. **Both Boruta and SFS scale linearly in $p$** for fixed iteration count. Boruta's constant is determined by $T \times T_{\text{RF}}$; SFS's is $k^* \times v \times T_{\mathcal{M}}$.

### What's Next

Guide 03 and the exercises cover making these methods scalable for $p > 300$: pre-screening, parallelisation, progressive subsampling, and memory-efficient data structures.

### Additional Resources

- Guide 02: Boruta theory, beam search, Optuna BOHB
- [BorutaPy](https://github.com/scikit-learn-contrib/boruta_py) — production Python Boruta implementation
- Kursa & Rudnicki (2010) — original Boruta paper
- [Optuna documentation](https://optuna.readthedocs.io)